# Test: Download ERA5 T2M (Jan 2026) from Earth Data Hub

Source: `era5-single-levels-atmosphere-daily-utc-v0` on EDH (DestinE).
Native resolution: **0.25°** (1440 × 721), already daily-averaged.

Output: `era5_t2m_dailymean_2026_p025_Jan_EDH.nc` next to your CDS p05 / p15 files,
so it can be regridded later for direct comparison.

**Kernel:** `destineE`.

In [ ]:
import os
import xarray as xr
import numpy as np

URL = "https://data.earthdatahub.destine.eu/era5/era5-single-levels-atmosphere-daily-utc-v0.zarr"
OUT_DIR = "/fs/ess/PYS1088/nachi/s2s_hindcast_verification/ERA5"
OUT_PATH = os.path.join(OUT_DIR, "era5_t2m_dailymean_2026_p025_Jan_EDH.nc")

TIME_START = "2026-01-01"
TIME_END   = "2026-01-31"

In [ ]:
# Sanity check: open the EDH public test dataset (no auth required).
# If this fails, the env / network / zarr-aiohttp wiring is the problem,
# not the PAT/auth. If this succeeds, move on to the auth'd ERA5 cell below.
test = xr.open_dataset(
    "https://data.earthdatahub.destine.eu/public/test-dataset-v0.zarr",
    chunks={},
    engine="zarr",
)
print("public test dataset OK")
print("  dims:", dict(test.sizes))
print("  vars:", list(test.data_vars))

In [ ]:
# EDH auth: uses a Personal Access Token from
#   https://earthdatahub.destine.eu/account-settings#my-personal-access-tokens
# Token is read from ~/.netrc — set up ONCE in your shell:
#   cat >> ~/.netrc <<'EOF'
#   machine data.earthdatahub.destine.eu
#       password <YOUR_PAT_HERE>
#   EOF
#   chmod 600 ~/.netrc
# Then trust_env=True (below) tells aiohttp to consult ~/.netrc automatically.
ds = xr.open_dataset(
    URL,
    storage_options={"client_kwargs": {"trust_env": True}},
    chunks={},
    engine="zarr",
    zarr_format=3,
)
print("dims:", dict(ds.sizes))
print("data_vars:", list(ds.data_vars))

In [ ]:
# ERA5 T2M is sometimes named '2t' (with a leading number → must use bracket access)
# and sometimes 't2m'. Pick whichever exists.
candidates = ["t2m"]
t2m_name = next((c for c in candidates if c in ds.data_vars), None)
if t2m_name is None:
    raise KeyError(f"No T2M variable found. Available: {list(ds.data_vars)}")
print(f"Using variable: {t2m_name!r}")

# Find the time-like dim (could be 'time', 'valid_time', 'date')
time_candidates = ["time", "valid_time", "date"]
time_dim = next((d for d in time_candidates if d in ds[t2m_name].dims), None)
if time_dim is None:
    raise KeyError(f"No time-like dim found. Dims of {t2m_name}: {ds[t2m_name].dims}")
print(f"Using time dim: {time_dim!r}")
print(f"Time coverage: {str(ds[time_dim].values.min())[:10]} -> {str(ds[time_dim].values.max())[:10]}")

In [ ]:
# Subset Jan 2026 and pull into memory (small: 31 × 721 × 1440 × 4B ≈ 130 MB)
t2m_jan = ds[t2m_name].sel({time_dim: slice(TIME_START, TIME_END)}).load()
print("shape:", t2m_jan.shape, "  dtype:", t2m_jan.dtype)
print("min/mean/max (K):",
      float(t2m_jan.min()), float(t2m_jan.mean()), float(t2m_jan.max()))

# Save as netCDF (rename to canonical 't2m' for downstream consistency with CDS files)
if t2m_name != "t2m":
    t2m_jan = t2m_jan.rename("t2m")
t2m_jan.to_netcdf(OUT_PATH)
print(f"Saved {OUT_PATH}  ({os.path.getsize(OUT_PATH)/1e6:.1f} MB)")

In [ ]:
# Produce 0.5° (p05) and 1.5° (p15) versions by subsampling at strides 2 and 6.
# Subsample (rather than coarsen().mean()) so the resulting lat/lon values
# line up EXACTLY with CDS p05 / p15 files — drop-in replacements.
import os

OUT_P05 = os.path.join(OUT_DIR, "era5_t2m_dailymean_2026_p05_Jan_EDH.nc")
OUT_P15 = os.path.join(OUT_DIR, "era5_t2m_dailymean_2026_p15_Jan_EDH.nc")

# t2m_jan is already in memory at 0.25° from the previous cell
t2m_p05 = t2m_jan.isel(latitude=slice(None, None, 2),  longitude=slice(None, None, 2))
t2m_p15 = t2m_jan.isel(latitude=slice(None, None, 6),  longitude=slice(None, None, 6))

print(f"p05 shape: {t2m_p05.shape}  lat[0,-1]={float(t2m_p05.latitude[0]):.2f},{float(t2m_p05.latitude[-1]):.2f}  lon[0,-1]={float(t2m_p05.longitude[0]):.2f},{float(t2m_p05.longitude[-1]):.2f}")
print(f"p15 shape: {t2m_p15.shape}  lat[0,-1]={float(t2m_p15.latitude[0]):.2f},{float(t2m_p15.latitude[-1]):.2f}  lon[0,-1]={float(t2m_p15.longitude[0]):.2f},{float(t2m_p15.longitude[-1]):.2f}")

t2m_p05.to_netcdf(OUT_P05)
t2m_p15.to_netcdf(OUT_P15)

print()
print(f"Saved {OUT_P05}  ({os.path.getsize(OUT_P05)/1e6:.1f} MB)")
print(f"Saved {OUT_P15}  ({os.path.getsize(OUT_P15)/1e6:.1f} MB)")
print(f"(p025 already saved: {OUT_PATH}  ({os.path.getsize(OUT_PATH)/1e6:.1f} MB))")

In [ ]:
# Quick visual sanity check: Jan 1 map
import matplotlib
matplotlib.use("Agg") if False else None   # leave inline for Jupyter
import matplotlib.pyplot as plt

check = xr.open_dataset(OUT_PATH).t2m.isel({time_dim: 0})
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.pcolormesh(check.longitude, check.latitude, check.values, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="K")
ax.set_title(f"EDH ERA5 T2M, day 1 of selection ({TIME_START})")
plt.tight_layout()
plt.show()